<a href="https://colab.research.google.com/github/Jeshani622/Data-Cleaning-Project/blob/main/Jeshani_Data_cleaning_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
file_path = '/content/messy_sales.csv'

# Check whether the file exists
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
else:
    print("File not found. Please check the file location.")

Dataset loaded successfully!


In [6]:
df.head()

,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
0,10001,Nikhil Jain,Mumbai,Laptop,3.0,50790.72,2026-01-18,Completed,61.0
1,10002,Simran Kaur,delhi,Monitor,1.0,10307.27,2026-01-28,Pending,50.0
2,10003,Zoya Ali,Delhi,Headphones,5.0,2439.64,2026-02-27,Delivered,35.0
3,10004,Priya Bansal,Delhi,Tablet,4.0,20945.65,02/03/2026,Pending,66.0
4,10005,NaN,Mumbai,Phone,4.0,15822.27,2026-02-14,Delivered,34.0


In [7]:
df.tail()

,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
180,10006,Priya Bansal,Delhi,Smartwatch,5.0,5768.41,2026-02-18,Completed,53.0
181,10026,Kabir Singh,Delhi,Keyboard,1.0,1506.37,15-02-2026,Cancelled,31.0
182,10081,Aarav Sharma,Bengaluru,Headphones,1.0,2591.0,2026-01-31,Pending,48.0
183,10121,Neha Gupta,Pune,Smartwatch,3.0,6517.3,2026-02-04,Cancelled,48.0
184,10151,Isha Roy,Mumbai,Smartwatch,3.0,6791.02,2026-01-15,Pending,20.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   OrderID       185 non-null    int64 
 1   CustomerName  183 non-null    object
 2   City          183 non-null    object
 3   Product       185 non-null    object
 4   Quantity      183 non-null    object
 5   UnitPrice     183 non-null    object
 6   OrderDate     183 non-null    object
 7   OrderStatus   183 non-null    object
 8   CustomerAge   183 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB


In [9]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset shape:", df.shape)

Number of rows: 185
Number of columns: 9
Dataset shape: (185, 9)


In [10]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

Missing values in each column:
OrderID         0
CustomerName    2
City            2
Product         0
Quantity        2
UnitPrice       2
OrderDate       2
OrderStatus     2
CustomerAge     2
dtype: int64

Total missing values:
14


In [11]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 5


In [12]:
# Remove spaces from column names
df.columns = df.columns.str.strip()

# Remove leading and trailing spaces from text values
text_columns = df.select_dtypes(include=['object', 'string']).columns

for col in text_columns:
    df[col] = df[col].str.strip()

print("Extra spaces removed successfully!")

display(df.head())

Extra spaces removed successfully!


,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
0,10001,Nikhil Jain,Mumbai,Laptop,3.0,50790.72,2026-01-18,Completed,61.0
1,10002,Simran Kaur,delhi,Monitor,1.0,10307.27,2026-01-28,Pending,50.0
2,10003,Zoya Ali,Delhi,Headphones,5.0,2439.64,2026-02-27,Delivered,35.0
3,10004,Priya Bansal,Delhi,Tablet,4.0,20945.65,02/03/2026,Pending,66.0
4,10005,NaN,Mumbai,Phone,4.0,15822.27,2026-02-14,Delivered,34.0


In [13]:
word_numbers = {
    'zero': 0,
    'one': 1,
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5,
    'six': 6,
    'seven': 7,
    'eight': 8,
    'nine': 9,
    'ten': 10,
    'forty': 40,
    'fifty': 50,
    'sixty': 60,
    'seventy': 70,
    'eighty': 80,
    'ninety': 90
}

def clean_number(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    # Convert number words
    if value in word_numbers:
        return word_numbers[value]

    # Remove commas
    value = value.replace(',', '')

    # Convert numerical values
    match = re.search(r'-?\d+\.?\d*', value)

    if match:
        return float(match.group())

    # Invalid values become missing
    return np.nan


print(clean_number("three"))
print(clean_number("2 pcs"))
print(clean_number("18,500"))
print(clean_number("unknown"))
print(clean_number("forty"))

3
2.0
18500.0
nan
40


In [14]:
numeric_columns = [
    'Quantity',
    'UnitPrice',
    'CustomerAge'
]

for col in numeric_columns:
    df[col] = df[col].apply(clean_number)

print("Numerical values cleaned successfully!")

Numerical values cleaned successfully!


In [15]:
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[numeric_columns].dtypes)

Quantity       float64
UnitPrice      float64
CustomerAge    float64
dtype: object


In [16]:
text_fill_columns = [
    'CustomerName',
    'City',
    'Product',
    'OrderStatus'
]

for col in text_fill_columns:
    df[col] = df[col].fillna('Unknown')

print("Missing text values handled!")

Missing text values handled!


In [17]:
for col in numeric_columns:

    median_value = df[col].median()

    if pd.notna(median_value):
        df[col] = df[col].fillna(median_value)
    else:
        print(f"Warning: {col} contains no valid numerical values.")

print("Missing numerical values handled!")

Missing numerical values handled!


In [18]:
outlier_columns = [
    'Quantity',
    'UnitPrice',
    'CustomerAge'
]

for col in outlier_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_limit) |
        (df[col] > upper_limit)
    ]

    print("\nColumn:", col)
    print("Lower limit:", lower_limit)
    print("Upper limit:", upper_limit)
    print("Number of outliers:", len(outliers))


Column: Quantity
Lower limit: -1.0
Upper limit: 7.0
Number of outliers: 2

Column: UnitPrice
Lower limit: -22699.199999999997
Upper limit: 44423.2
Number of outliers: 18

Column: CustomerAge
Lower limit: -8.0
Upper limit: 96.0
Number of outliers: 1


In [19]:
# Handle unrealistic customer ages
df.loc[
    (df['CustomerAge'] < 1) |
    (df['CustomerAge'] > 100),
    'CustomerAge'
] = np.nan

# Handle invalid quantities
df.loc[
    df['Quantity'] <= 0,
    'Quantity'
] = np.nan

# Handle invalid prices
df.loc[
    df['UnitPrice'] <= 0,
    'UnitPrice'
] = np.nan


# Replace statistical outliers with median
for col in outlier_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    median_value = df[col].median()

    if pd.notna(median_value):
        df.loc[
            (df[col] < lower_limit) |
            (df[col] > upper_limit),
            col
        ] = median_value

    df[col] = df[col].fillna(median_value)

print("Outliers handled successfully!")

Outliers handled successfully!


In [20]:
df['CustomerName'] = df['CustomerName'].str.title()

df['City'] = df['City'].str.title()

df['Product'] = df['Product'].str.title()

df['OrderStatus'] = df['OrderStatus'].str.title()

print("Text standardization completed!")

print("\nUnique Cities:")
print(df['City'].unique())

print("\nUnique Order Statuses:")
print(df['OrderStatus'].unique())

Text standardization completed!

Unique Cities:
['Mumbai' 'Delhi' 'Chennai' 'Bengaluru' 'Pune' 'Kolkata' 'Gurugram'
 'Noida' 'Unknown']

Unique Order Statuses:
['Completed' 'Pending' 'Delivered' 'Shipped' 'Cancelled' 'Unknown']


In [21]:
df['OrderDate'] = pd.to_datetime(
    df['OrderDate'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Date cleaning completed!")

display(df[['OrderDate']].head(10))

Date cleaning completed!


,OrderDate
0,2026-01-18
1,2026-01-28
2,2026-02-27
3,2026-03-02
4,2026-02-14
5,2026-02-18
6,2026-01-30
7,2026-03-23
8,2026-03-29
9,2026-02-04


In [22]:
df['OrderDate'] = df['OrderDate'].dt.strftime('%Y-%m-%d')

print(df['OrderDate'].head(10))

0    2026-01-18
1    2026-01-28
2    2026-02-27
3    2026-03-02
4    2026-02-14
5    2026-02-18
6    2026-01-30
7    2026-03-23
8    2026-03-29
9    2026-02-04
Name: OrderDate, dtype: object


In [23]:
# Check duplicates before removing
duplicates_before = df.duplicated().sum()

print("Duplicates before cleaning:", duplicates_before)

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

# Check duplicates after removing
duplicates_after = df.duplicated().sum()

print("Duplicates after cleaning:", duplicates_after)

Duplicates before cleaning: 5
Duplicates after cleaning: 0


In [24]:
# OrderID as string
df['OrderID'] = df['OrderID'].astype('string')

# Quantity as integer
df['Quantity'] = df['Quantity'].round().astype('Int64')

# CustomerAge as integer
df['CustomerAge'] = df['CustomerAge'].round().astype('Int64')

# UnitPrice as float
df['UnitPrice'] = df['UnitPrice'].astype(float)

# Text columns as string
string_columns = [
    'CustomerName',
    'City',
    'Product',
    'OrderStatus'
]

for col in string_columns:
    df[col] = df[col].astype('string')

# Convert OrderDate back to datetime
df['OrderDate'] = pd.to_datetime(
    df['OrderDate'],
    errors='coerce'
)

print("Data types converted successfully!")

df.info()

Data types converted successfully!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   OrderID       180 non-null    string        
 1   CustomerName  180 non-null    string        
 2   City          180 non-null    string        
 3   Product       180 non-null    string        
 4   Quantity      180 non-null    Int64         
 5   UnitPrice     180 non-null    float64       
 6   OrderDate     177 non-null    datetime64[ns]
 7   OrderStatus   180 non-null    string        
 8   CustomerAge   180 non-null    Int64         
dtypes: Int64(2), datetime64[ns](1), float64(1), string(5)
memory usage: 13.1 KB


In [25]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print("TotalAmount column created successfully!")

display(df.head())

TotalAmount column created successfully!


,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge,TotalAmount
0,10001,Nikhil Jain,Mumbai,Laptop,3,10451.145,2026-01-18,Completed,61,31353.435
1,10002,Simran Kaur,Delhi,Monitor,1,10307.270,2026-01-28,Pending,50,10307.27
2,10003,Zoya Ali,Delhi,Headphones,5,2439.640,2026-02-27,Delivered,35,12198.2
3,10004,Priya Bansal,Delhi,Tablet,4,20945.650,2026-03-02,Pending,66,83782.6
4,10005,Unknown,Mumbai,Phone,4,15822.270,2026-02-14,Delivered,34,63289.08


In [26]:
print("Final dataset shape:", df.shape)

print("Total rows:", len(df))

print("Total columns:", len(df.columns))

Final dataset shape: (180, 10)
Total rows: 180
Total columns: 10


In [27]:
print("Missing values after cleaning:")

print(df.isnull().sum())

Missing values after cleaning:
OrderID         0
CustomerName    0
City            0
Product         0
Quantity        0
UnitPrice       0
OrderDate       3
OrderStatus     0
CustomerAge     0
TotalAmount     0
dtype: int64


In [28]:
print("Remaining duplicate rows:", df.duplicated().sum())

Remaining duplicate rows: 0


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   OrderID       180 non-null    string        
 1   CustomerName  180 non-null    string        
 2   City          180 non-null    string        
 3   Product       180 non-null    string        
 4   Quantity      180 non-null    Int64         
 5   UnitPrice     180 non-null    float64       
 6   OrderDate     177 non-null    datetime64[ns]
 7   OrderStatus   180 non-null    string        
 8   CustomerAge   180 non-null    Int64         
 9   TotalAmount   180 non-null    Float64       
dtypes: Float64(1), Int64(2), datetime64[ns](1), float64(1), string(5)
memory usage: 14.7 KB


In [30]:
display(df.head(10))

,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge,TotalAmount
0,10001,Nikhil Jain,Mumbai,Laptop,3,10451.145,2026-01-18,Completed,61,31353.435
1,10002,Simran Kaur,Delhi,Monitor,1,10307.270,2026-01-28,Pending,50,10307.27
2,10003,Zoya Ali,Delhi,Headphones,5,2439.640,2026-02-27,Delivered,35,12198.2
3,10004,Priya Bansal,Delhi,Tablet,4,20945.650,2026-03-02,Pending,66,83782.6
4,10005,Unknown,Mumbai,Phone,4,15822.270,2026-02-14,Delivered,34,63289.08
5,10006,Priya Bansal,Delhi,Smartwatch,5,5768.410,2026-02-18,Completed,53,28842.05
6,10007,Meera Joshi,Chennai,Headphones,1,2159.370,2026-01-30,Shipped,23,2159.37
7,10008,Sara Thomas,Bengaluru,Phone,3,16801.060,2026-03-23,Shipped,28,50403.18
8,10009,Tara Kapoor,Mumbai,Headphones,3,2651.370,2026-03-29,Completed,56,7954.11
9,10010,Nikhil Jain,Pune,Headphones,2,2471.700,2026-02-04,Delivered,32,4943.4


In [31]:
df.describe()

,Quantity,UnitPrice,OrderDate,CustomerAge,TotalAmount
count,180.0,180.000000,177,180.0,180.0
mean,2.905556,9969.275361,2026-02-18 18:18:18.305084672,44.177778,29677.557444
min,1.0,0.010000,2026-01-01 00:00:00,3.0,0.04
25%,2.0,2467.500000,2026-01-28 00:00:00,31.0,5735.985
50%,3.0,10451.145000,2026-02-21 00:00:00,45.0,21768.805
75%,4.0,16355.605000,2026-03-12 00:00:00,57.0,51416.4125
max,5.0,24768.970000,2026-03-31 00:00:00,70.0,101081.95
std,1.413017,7293.146611,NaN,15.312446,26843.685428


In [32]:
print("Cities:")
print(df['City'].unique())

print("\nOrder Statuses:")
print(df['OrderStatus'].unique())

print("\nProducts:")
print(df['Product'].unique())

Cities:
<StringArray>
[   'Mumbai',     'Delhi',   'Chennai', 'Bengaluru',      'Pune',   'Kolkata',
  'Gurugram',     'Noida',   'Unknown']
Length: 9, dtype: string

Order Statuses:
<StringArray>
['Completed', 'Pending', 'Delivered', 'Shipped', 'Cancelled', 'Unknown']
Length: 6, dtype: string

Products:
<StringArray>
[    'Laptop',    'Monitor', 'Headphones',     'Tablet',      'Phone',
 'Smartwatch',   'Keyboard',      'Mouse']
Length: 8, dtype: string


In [33]:
output_path = '/content/cleaned_sales_dataset.csv'

df.to_csv(output_path, index=False)

print("Cleaned CSV file saved successfully!")
print("File location:", output_path)

Cleaned CSV file saved successfully!
File location: /content/cleaned_sales_dataset.csv


In [34]:
from google.colab import files

files.download('/content/cleaned_sales_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Mini Project Summary**

Objective: The objective of this project was to clean a messy sales dataset using Python and Pandas.

Cleaning operations performed:
1. Data inspection using head(), info(), shape and isnull().
2. Removed unnecessary spaces from column names and text values.
3. Handled missing values using median replacement and the Unknown category.
4. Corrected numerical data type errors such as three, 2 pcs, 18,500 and forty.
5. Identified outliers using the IQR method and replaced them with median values.
6. Standardized text values such as city names and order statuses.
7. Converted different date formats into a standard format.
8. Removed duplicate records.
9. Converted columns into suitable data types.
10. Created the TotalAmount column using Quantity * UnitPrice.
11. Exported the cleaned dataset as cleaned_sales_dataset.csv.

Conclusion: The cleaning process improved the consistency, accuracy and quality of the sales dataset. The final dataset is ready for further analysis.